# 리뷰 텍스트 전처리 - Step 2: 중복 처리

`reviews_step1_cleaned.csv`를 입력으로 받아 같은 작성자의 다중 리뷰 작성을 식별하고 처리한다.

## 처리 전략

| Step | 조건 | 처리 |
|------|------|------|
| 0 | 작성자 = `'-'` (탈퇴/닉변) | 각 행을 고유 ID로 분리 (서로 다른 사람으로 처리) |
| 1 | 리뷰타입 = `month` | 다중 작성 검토에서 제외 (다른 시점의 정보) |
| 2-A | 같은 작성자 + 같은 상품 + **같은 옵션** + 같은 타입 + 24h 이내 | 한글_글자수 많은 1개 보존 |
| 2-B | 같은 작성자 + 같은 상품 + **다른 옵션** + 같은 타입 + 24h 이내 + 코사인 유사도 ≥ 0.85 | 1개 보존, `multi_option_dup` 플래그 |
| 2-C | 같은 작성자 + 같은 상품 + **다른 옵션** + 같은 타입 + 24h 이내 + 코사인 유사도 < 0.85 | 둘 다 보존, `multi_option_unique` 플래그 |
| 3 | 다른 타입 + 24h 이내 + 코사인 유사도 ≥ 0.85 | 1개 보존, `multi_type_dup` 플래그 |
| 4 | 다른 타입 + 24h 이내 + 코사인 유사도 < 0.85 | 둘 다 보존, `multi_type_unique` 플래그 |
| 5 | 24h 초과 | 재구매 분류, `is_repurchase` 플래그 |

**코사인 유사도**: TF-IDF 문자 bi-gram 벡터 기반. 텍스트 길이 15자 미만이면 우연 일치 방지를 위해 완전일치만 인정.

**24h 세션 분리**: 같은 (작성자, 상품) 안에서 작성일 정렬 후 인접 리뷰 간 시간차 ≤ 24h면 동일 세션으로 묶음.

---
## 0. 데이터 로드

In [1]:
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

INPUT_PATH = "reviews_step1_cleaned.csv"
TEXT_COL = "리뷰내용_clean"

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f"로드 완료: {len(df):,}건")
print(f"컬럼 수: {df.shape[1]}개")

로드 완료: 685,042건
컬럼 수: 49개


In [2]:
# 작성일 datetime 변환
df['작성일'] = pd.to_datetime(df['작성일'], errors='coerce')

# 작성일 결측 확인
n_nat = df['작성일'].isna().sum()
print(f"작성일 결측: {n_nat:,}건")

# 텍스트 결측 → 빈 문자열로
df[TEXT_COL] = df[TEXT_COL].fillna('').astype(str)

작성일 결측: 0건


---
## STEP 0. 탈퇴/닉변 작성자 처리

작성자가 `'-'`인 경우, 무신사 시스템상 익명 처리된 케이스이므로 각 행을 **서로 다른 사람**으로 취급한다.
→ 이후 중복 검토에서 자동으로 제외됨.

In [3]:
# '-' 작성자에 고유 ID 부여
df['작성자_norm'] = df['작성자'].astype(str)
mask_anon = df['작성자_norm'].str.strip() == "'-"

# 익명 행은 인덱스 기반 고유 ID로 변환
df.loc[mask_anon, '작성자_norm'] = (
    '_anon_' + df.loc[mask_anon].index.astype(str)
)

print(f"탈퇴/닉변 작성자: {mask_anon.sum():,}건 → 고유 ID로 변환")
print(f"고유 작성자 수: {df['작성자_norm'].nunique():,}명")

탈퇴/닉변 작성자: 3,975건 → 고유 ID로 변환
고유 작성자 수: 349,096명


---
## STEP 1. month 타입 분리

리뷰타입이 `month`인 리뷰는 한 달 사용 후 작성하는 후기로, 다른 타입(일반/스타일)과 시점·결이 다르다.
→ 중복 검토 대상에서 제외하고 그대로 보존.

In [4]:
print("리뷰타입 분포:")
print(df['리뷰타입'].value_counts(dropna=False))

mask_month = df['리뷰타입'] == 'month'
df_month = df[mask_month].copy()
df_main = df[~mask_month].copy()

print(f"\nmonth 리뷰: {len(df_month):,}건 (중복 검토 제외)")
print(f"main 리뷰: {len(df_main):,}건 (중복 검토 대상)")

리뷰타입 분포:
리뷰타입
goods         376942
photo         109292
style         104397
general        68881
month          25088
experience       442
Name: count, dtype: int64

month 리뷰: 25,088건 (중복 검토 제외)
main 리뷰: 659,954건 (중복 검토 대상)


---
## 보조 컬럼 + 코사인 유사도 함수

- **옵션키**: `(구매사이즈, 구매상세)` 튜플. NaN은 `None`으로 통일하여 옵션 정보 없는 행끼리 동일 옵션으로 인정.
- **보존 기준**: Step 1에서 만들어진 `한글_글자수` 컬럼을 그대로 활용. 그룹 내에서 한글 글자가 많은 리뷰가 base.
- **코사인 유사도**: TF-IDF 문자 bi-gram 벡터 기반. 비교 전 공백·문장부호를 제거하는 정규화를 적용해 마침표/느낌표 한두 개 차이로 유사도가 깎이는 문제 방지. 짧은 텍스트(15자 미만)는 완전일치만 인정.

In [5]:
# 옵션키 생성: (사이즈, 상세) 튜플, NaN은 None
def make_option_key(s, d):
    s_v = s if pd.notna(s) else None
    d_v = d if pd.notna(d) else None
    return (s_v, d_v)

df_main['옵션키'] = [
    make_option_key(s, d)
    for s, d in zip(df_main['구매사이즈'], df_main['구매상세'])
]

print(f"옵션키 고유값: {df_main['옵션키'].nunique():,}개")
print(f"한글_글자수 통계:\n{df_main['한글_글자수'].describe()}")

옵션키 고유값: 1,570개
한글_글자수 통계:
count    659954.000000
mean         33.483250
std          21.750261
min           0.000000
25%          22.000000
50%          27.000000
75%          36.000000
max        1051.000000
Name: 한글_글자수, dtype: float64


In [6]:
# 코사인 유사도 (TF-IDF 문자 bi-gram 기반)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

COSINE_THRESHOLD = 0.85
MIN_LEN_FOR_COSINE = 15  # 짧은 텍스트는 완전일치만 인정

def normalize_for_compare(s: str) -> str:
    """코사인 유사도 비교 전용 정규화: 공백·문장부호 제거.
    문장부호/마침표 한두 개 차이로 유사도가 깎이는 문제를 방지한다."""
    return re.sub(r'[\s\.\,\!\?\~\^\;\:\-\_\(\)\[\]\"\']+', '', s)

def char_bigram_analyzer(s: str):
    """TfidfVectorizer용 문자 bi-gram 분석기."""
    return [s[i:i+2] for i in range(len(s) - 1)] if len(s) >= 2 else [s]

def cosine_sim(s1: str, s2: str) -> float:
    """TF-IDF 문자 bi-gram 코사인 유사도. 정규화(공백/문장부호 제거) 후 계산.
    짧은 텍스트(15자 미만)는 완전일치만 1.0으로 인정."""
    if not s1 or not s2:
        return 0.0
    n1 = normalize_for_compare(s1)
    n2 = normalize_for_compare(s2)
    if len(n1) < MIN_LEN_FOR_COSINE or len(n2) < MIN_LEN_FOR_COSINE:
        return 1.0 if n1 == n2 else 0.0
    try:
        vec = TfidfVectorizer(analyzer=char_bigram_analyzer)
        tfidf = vec.fit_transform([n1, n2])
        return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])
    except Exception:
        return 0.0

# 동작 확인
print("[코사인 유사도 함수 테스트]")
print(f"문장부호만 다름:    {cosine_sim('정말 좋아요. 잘 입을게요!', '정말 좋아요 잘 입을게요'):.3f}")
print(f"공백만 다름:        {cosine_sim('가성비 좋고 핏도 정사이즈입니다 추천드려요', '가성비좋고핏도정사이즈입니다추천드려요'):.3f}")
print(f"부분 유사:          {cosine_sim('두께감 적당하고 핏 좋습니다 추천드려요', '두께감 적당하고 색감 예쁘네요 추천'):.3f}")
print(f"완전 다름:          {cosine_sim('정말 좋아요 잘 입을게요', '사이즈가 너무 작아서 환불했습니다'):.3f}")
print(f"짧은 텍스트 동일:   {cosine_sim('좋아요', '좋아요'):.3f}")
print(f"짧은 텍스트 다름:   {cosine_sim('좋아요', '괜찮아요'):.3f}")

[Jaccard 함수 테스트]
문장부호만 다름:    1.000
공백만 다름:        1.000
부분 유사:          0.304
완전 다름:          0.000
짧은 텍스트 동일:   1.000
짧은 텍스트 다름:   0.000


---
## 24h 세션 분리

같은 (작성자, 상품) 안에서 작성일 순 정렬 → 인접 리뷰 간 시간차 ≤ 24h면 같은 세션 ID 부여.
서로 다른 세션은 **재구매 케이스(Step 5)** 로 분류된다.

In [7]:
# 정렬: 작성자 → 상품 → 작성일
df_main = df_main.sort_values(
    ['작성자_norm', 'goodsNo', '작성일']
).reset_index(drop=True)

# 그룹별 세션 ID 부여 (벡터화)
# 같은 (작성자, 상품) 안에서 인접 행 시간차가 24h 초과면 새 세션 시작
g = df_main.groupby(['작성자_norm', 'goodsNo'])
prev_time = g['작성일'].shift(1)
time_diff_h = (df_main['작성일'] - prev_time).dt.total_seconds() / 3600

# 새 세션 시작 시점 = 그룹 첫 행 OR 시간차 > 24h
new_session = (prev_time.isna()) | (time_diff_h > 24)
df_main['세션'] = new_session.groupby(
    [df_main['작성자_norm'], df_main['goodsNo']]
).cumsum().astype(int)

print(f"세션 부여 완료")
print(f"(작성자, 상품) 그룹 수: {g.ngroups:,}")
print(f"(작성자, 상품, 세션) 그룹 수: {df_main.groupby(['작성자_norm', 'goodsNo', '세션']).ngroups:,}")

세션 부여 완료
(작성자, 상품) 그룹 수: 469,991
(작성자, 상품, 세션) 그룹 수: 526,828


In [8]:
#pd.set_option('display.max_columns',None)
pd.options.display.max_columns = 30

In [9]:
# 같은 (작성자, 상품)에 세션이 2개 이상 → 재구매
session_per_pair = df_main.groupby(['작성자_norm', 'goodsNo'])['세션'].transform('nunique')
df_main['is_repurchase'] = session_per_pair > 1

n_repurchase_rows = df_main['is_repurchase'].sum()
n_repurchase_pairs = (session_per_pair > 1).groupby(
    [df_main['작성자_norm'], df_main['goodsNo']]
).first().sum()
print(f"재구매 행 수: {n_repurchase_rows:,}건")
print(f"재구매 (작성자, 상품) 쌍: {n_repurchase_pairs:,}쌍")

재구매 행 수: 129,495건
재구매 (작성자, 상품) 쌍: 50,181쌍


---
## 그룹 단위 중복 처리 (Step 2 ~ 4)

`(작성자_norm, goodsNo, 세션)` 단위로 묶고:

1. 그룹 크기 = 1 → 단독 리뷰 (그대로 keep)
2. 그룹 크기 ≥ 2:
   - 한글_글자수가 가장 많은 행을 **기준 리뷰**로 지정
   - 같은 그룹의 다른 리뷰 각각에 대해 옵션 일치 여부, 타입 일치 여부, 코사인 유사도를 계산하여 케이스 분기

In [10]:
def process_group(group: pd.DataFrame) -> pd.DataFrame:
    """
    같은 (작성자, 상품, 세션) 그룹 내 다중 리뷰 처리.

    반환 컬럼:
      - action: 'keep' | 'drop'
      - dup_flag: 'unique' | 'same_option_same_type_dup' | 'multi_option_dup' |
                  'multi_option_unique' | 'multi_type_dup' | 'multi_type_unique' |
                  'multi_both_dup' | 'multi_both_unique'
                  (base와 동반자 모두 같은 플래그를 공유. 구분은 is_base로)
      - is_base: 그룹의 기준 리뷰 여부 (단독 그룹은 모두 True)
      - kept_id: 그룹의 base 리뷰번호 (drop된 경우 어느 리뷰가 보존되었는지 추적용)
    """
    g = group.copy()

    if len(g) == 1:
        g['action'] = 'keep'
        g['dup_flag'] = 'unique'
        g['is_base'] = True
        g['kept_id'] = g['리뷰번호'].iloc[0]
        return g

    # 한글_글자수 내림차순 + 작성일 오름차순 (동률 시 먼저 작성된 게 기준)
    g = g.sort_values(['한글_글자수', '작성일'], ascending=[False, True])

    base_idx = g.index[0]
    base_row = g.loc[base_idx]
    base_text = base_row[TEXT_COL]
    base_option = base_row['옵션키']
    base_type = base_row['리뷰타입']
    base_review_id = base_row['리뷰번호']

    actions = ['keep']  # 기준 리뷰는 항상 keep
    flags = [None]      # 기준 리뷰 플래그는 추후 결정

    for idx in g.index[1:]:
        row = g.loc[idx]
        text = row[TEXT_COL]
        option = row['옵션키']
        rtype = row['리뷰타입']

        same_opt = (option == base_option)
        same_typ = (rtype == base_type)
        sim = cosine_sim(base_text, text)

        if same_opt and same_typ:
            # Step 2-A: 같은 옵션 + 같은 타입 → 무조건 drop
            actions.append('drop')
            flags.append('same_option_same_type_dup')
        elif (not same_opt) and same_typ:
            # Step 2-B/C: 다른 옵션 + 같은 타입
            if sim >= COSINE_THRESHOLD:
                actions.append('drop')
                flags.append('multi_option_dup')
            else:
                actions.append('keep')
                flags.append('multi_option_unique')
        elif same_opt and (not same_typ):
            # Step 3/4: 같은 옵션 + 다른 타입
            if sim >= COSINE_THRESHOLD:
                actions.append('drop')
                flags.append('multi_type_dup')
            else:
                actions.append('keep')
                flags.append('multi_type_unique')
        else:
            # 다른 옵션 + 다른 타입
            if sim >= COSINE_THRESHOLD:
                actions.append('drop')
                flags.append('multi_both_dup')
            else:
                actions.append('keep')
                flags.append('multi_both_unique')

    g['action'] = actions
    g['dup_flag'] = flags

    # base의 dup_flag: 그룹 내에서 발생한 가장 강한 관계로 표시 (동반자와 같은 플래그)
    # base/동반자 구분은 is_base 컬럼으로 한다 (kept_ 접두사 사용 안 함).
    non_none_flags = [f for f in flags[1:] if f is not None]
    if non_none_flags:
        dup_flags = [f for f in non_none_flags if f.endswith('_dup')]
        g.loc[base_idx, 'dup_flag'] = dup_flags[0] if dup_flags else non_none_flags[0]
    else:
        g.loc[base_idx, 'dup_flag'] = 'unique'

    # is_base: base 행만 True
    g['is_base'] = False
    g.loc[base_idx, 'is_base'] = True
    g['kept_id'] = base_review_id
    return g

In [11]:
# 그룹 사이즈 사전 확인
group_sizes = df_main.groupby(['작성자_norm', 'goodsNo', '세션']).size()
print(f"전체 (작성자, 상품, 세션) 그룹 수: {len(group_sizes):,}")
print(f"  단일 리뷰 그룹: {(group_sizes == 1).sum():,}")
print(f"  다중 리뷰 그룹(≥2): {(group_sizes >= 2).sum():,}")
print(f"\n다중 리뷰 그룹 크기 분포:")
print(group_sizes[group_sizes >= 2].value_counts().sort_index().head(10))

전체 (작성자, 상품, 세션) 그룹 수: 526,828
  단일 리뷰 그룹: 425,294
  다중 리뷰 그룹(≥2): 101,534

다중 리뷰 그룹 크기 분포:
2    71897
3    28472
4      734
5      102
6      315
7        5
8        2
9        7
Name: count, dtype: int64


In [12]:
# 단일 그룹 / 다중 그룹 분리 처리 (성능 최적화)
df_main = df_main.merge(
    group_sizes.rename('그룹크기').reset_index(),
    on=['작성자_norm', 'goodsNo', '세션'],
    how='left'
)

mask_single = df_main['그룹크기'] == 1
df_single = df_main[mask_single].copy()
df_multi = df_main[~mask_single].copy()

print(f"단일 그룹 행: {len(df_single):,}")
print(f"다중 그룹 행: {len(df_multi):,}")

# 단일 그룹: 일괄 처리 (빠름)
df_single['action'] = 'keep'
df_single['dup_flag'] = 'unique'
df_single['is_base'] = True
df_single['kept_id'] = df_single['리뷰번호']

# 다중 그룹: process_group 적용 (오래 걸림)
tqdm.pandas(desc="다중 그룹 처리")
df_multi_processed = (
    df_multi.groupby(['작성자_norm', 'goodsNo', '세션'], group_keys=False, sort=False)
            .progress_apply(process_group)
)

# pandas 버전 호환: groupby 키 컬럼이 결과에서 빠진 경우(pandas 3.0+) 인덱스 기반으로 복구
# (process_group 내부에서 원본 정수 인덱스가 보존되므로 가능)
for col in ['작성자_norm', 'goodsNo', '세션']:
    if col not in df_multi_processed.columns:
        df_multi_processed[col] = df_multi.loc[df_multi_processed.index, col]

# 합치기
df_main_processed = pd.concat(
    [df_single, df_multi_processed], ignore_index=True
).sort_values(['작성자_norm', 'goodsNo', '작성일']).reset_index(drop=True)

print(f"\n처리 완료: {len(df_main_processed):,}건")

단일 그룹 행: 425,294
다중 그룹 행: 234,660


다중 그룹 처리:   0%|          | 0/101534 [00:00<?, ?it/s]


처리 완료: 659,954건


---
## 처리 결과 통계

In [13]:
print("[action 분포]")
print(df_main_processed['action'].value_counts())
print(f"\n[dup_flag 분포]")
print(df_main_processed['dup_flag'].value_counts(dropna=False))
print(f"\n[is_repurchase 분포]")
print(df_main_processed['is_repurchase'].value_counts())

[action 분포]
action
keep    602367
drop     57587
Name: count, dtype: int64

[dup_flag 분포]
dup_flag
unique                       425294
multi_type_unique            126016
multi_type_dup                94935
multi_option_unique            6839
multi_option_dup               3520
multi_both_unique              1422
same_option_same_type_dup      1291
multi_both_dup                  637
Name: count, dtype: int64

[is_repurchase 분포]
is_repurchase
False    530459
True     129495
Name: count, dtype: int64


In [14]:
# 샘플 확인: drop된 리뷰 vs 보존된 기준 리뷰 페어
print("[multi_option_dup 샘플]")
sample_dup = df_main_processed[
    (df_main_processed['dup_flag'] == 'multi_option_dup') &
    (df_main_processed['action'] == 'drop')
].head(5)
for _, row in sample_dup.iterrows():
    base = df_main_processed[df_main_processed['리뷰번호'] == row['kept_id']].iloc[0]
    print(f"\n  [기준] {row['kept_id']} | 옵션={base['옵션키']} | 한글_글자수={base['한글_글자수']}")
    print(f"     → {base[TEXT_COL][:80]}")
    print(f"  [drop] {row['리뷰번호']} | 옵션={row['옵션키']} | 한글_글자수={row['한글_글자수']}")
    print(f"     → {row[TEXT_COL][:80]}")

[multi_option_dup 샘플]

  [기준] 15343006 | 옵션=('M', '셔츠딥그레이') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합
  [drop] 15342958 | 옵션=('M', '셔츠라이트베이지') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합

  [기준] 15343006 | 옵션=('M', '셔츠딥그레이') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합
  [drop] 15342623 | 옵션=('M', '셔츠블랙') | 한글_글자수=71
     → 셋업에 벌룬핏이라 색상별로 구매했습니다 두께감은 일반청자켓 셋업 정도로 생각하시면 되고 한벌 또는 셔츠와 팬츠를 별로도 입을수있어 환절기에 적합

  [기준] 42693190 | 옵션=('2XL', '베이지') | 한글_글자수=43
     → 옷감이 가벼워 여름에 시원하게 입을수 있겠네요뒷주머니가 사진에 안보였는데 진짜로 없어 좀 아쉽네요;;
  [drop] 42693216 | 옵션=('M', '베이지') | 한글_글자수=43
     → 옷감이 가벼워 여름에 시원하게 입을수 있겠네요뒷주머니가 사진에 안보였는데 진짜로 없어 좀 아쉽네요;;

  [기준] 44590746 | 옵션=('XL', '차콜') | 한글_글자수=27
     → 다른색들도 다 맘에들어서 더 샀는데 역시 맘에드네요 감사합니다ㅎㅎ
  [drop] 44590756 | 옵션=('XL', '블랙') | 한글_글자수=27
     → 다른색들도 다 맘에들어서 더 샀는데 역시 맘에드네요 감사합니다ㅎㅎ

  [기준] 44590746

In [15]:
print("[multi_option_unique 샘플 - 둘 다 보존되는 케이스]")
# 동반자(is_base=False)만 순회 → 페어가 한 번씩만 출력됨
companions = (df_main_processed[
    (df_main_processed['dup_flag'] == 'multi_option_unique') & (~df_main_processed['is_base'])
].head(5))
for _, row in companions.iterrows():
    base = df_main_processed[df_main_processed['리뷰번호'] == row['kept_id']].iloc[0]
    print(f"\n  [기준]      {row['kept_id']} | 옵션={base['옵션키']} | 한글_글자수={base['한글_글자수']}")
    print(f"     → {base[TEXT_COL][:100]}")
    print(f"  [같이 보존] {row['리뷰번호']} | 옵션={row['옵션키']} | 한글_글자수={row['한글_글자수']}")
    print(f"     → {row[TEXT_COL][:100]}")

[multi_option_unique 샘플 - 둘 다 보존되는 케이스]

  [기준]      53544349 | 옵션=('M', None) | 한글_글자수=56
     → 먼저 배송이 너무 빨라서 좋았어요! 아주 빠르게 와서 크리스마스 연말 커플룩으로 입을 수 있었어요 :-) 기본 맨투맨으로 입기 딱인 재질입니다!
  [같이 보존] 53544370 | 옵션=('L', None) | 한글_글자수=33
     → 빠른배송이 제일 마음에 들었어요. 목부분 손목부분 쫀쫀해서 그것도 좋았습니당

  [기준]      446800 | 옵션=('M', '블랙') | 한글_글자수=41
     → 타브랜드(XXX벨)처럼 후드가 두툼하고 가격대비 퀄리티가 좋아서 만족스럽고 가격이 정말 마음에듭니다.
  [같이 보존] 446799 | 옵션=('M', '딥그린') | 한글_글자수=32
     → 타브랜드(XXX벨)처럼 후드가 두툼하고 가격대비 퀄리티가 좋아서 만족스럽습니다.

  [기준]      28940102 | 옵션=('LARGE', None) | 한글_글자수=25
     → 좋아요 두꺼운편인듯 사이즈는 좋아요 환절기에 입어야 할듯
  [같이 보존] 28940077 | 옵션=('MEDIUM', None) | 한글_글자수=21
     → 좋아요 근데 얇지 않습니다 좀 두꺼운편인거 같아요

  [기준]      4753204 | 옵션=('L', '블랙') | 한글_글자수=37
     → 색상 다양하게 구매했는데 전부 마음에 들어요 딱 무난해서 데일리로 잘 입고 다니고 있어요
  [같이 보존] 4753194 | 옵션=('L', '라임') | 한글_글자수=33
     → 색이 너무 마음에 들어요! 쨍한 색감의 티가 갖고 싶었는데 이번기회에 마련했어요

  [기준]      47736657 | 옵션=('L', '네이비') | 한글_글자수=44
     → 일단 옷 재질이 너무 좋습니다 가격대 치고 디자인도 그렇고 두께감도 엄청 두꺼워요 가성비 지리는 옷 입니다
  [같

---
## month 리뷰 통합 + 저장

- `reviews_step2_dedup.csv`: 최종 보존 리뷰 (Step 3 임베딩 입력)
- `reviews_step2_dropped.csv`: drop된 리뷰 (검증용 보관)

In [16]:
# month 리뷰는 그대로 keep, 플래그만 추가
df_month['action'] = 'keep'
df_month['dup_flag'] = 'month_excluded'
df_month['is_base'] = True
df_month['kept_id'] = df_month['리뷰번호']
df_month['is_repurchase'] = False
df_month['세션'] = 0
df_month['그룹크기'] = 1
df_month['옵션키'] = [
    make_option_key(s, d)
    for s, d in zip(df_month['구매사이즈'], df_month['구매상세'])
]

# 컬럼 정렬을 df_main_processed에 맞춤
df_month = df_month[df_main_processed.columns]

# 통합
df_final = pd.concat([df_main_processed, df_month], ignore_index=True)
print(f"통합 후: {len(df_final):,}건")

통합 후: 685,042건


In [17]:
# 최종 저장: keep만
df_keep = df_final[df_final['action'] == 'keep'].copy()
df_drop = df_final[df_final['action'] == 'drop'].copy()

print(f"최종 보존: {len(df_keep):,}건 ({len(df_keep)/len(df_final)*100:.2f}%)")
print(f"제거된 중복: {len(df_drop):,}건 ({len(df_drop)/len(df_final)*100:.2f}%)")

print(f"\n[보존 리뷰 dup_flag 분포]")
print(df_keep['dup_flag'].value_counts())

print(f"\n[제거된 리뷰 dup_flag 분포]")
print(df_drop['dup_flag'].value_counts())

최종 보존: 627,455건 (91.59%)
제거된 중복: 57,587건 (8.41%)

[보존 리뷰 dup_flag 분포]
dup_flag
unique                       425294
multi_type_unique            126016
multi_type_dup                40544
month_excluded                25088
multi_option_unique            6839
multi_option_dup               1585
multi_both_unique              1422
same_option_same_type_dup       611
multi_both_dup                   56
Name: count, dtype: int64

[제거된 리뷰 dup_flag 분포]
dup_flag
multi_type_dup               54391
multi_option_dup              1935
same_option_same_type_dup      680
multi_both_dup                 581
Name: count, dtype: int64


#### 플래그 작명 규칙 : multi_{무엇이 다른가}_{판정결과}
- 무엇이 다른가 : option(옵션만 다름) / type(타입만 다름) / both(둘 다 다름)
- 판정결과 : _dup (코사인 유사도 >= 0.85, 같은 후기로 판정) / _unique(코사인 유사도 < 0.85, 다른 후기로 판정)

### 보존리뷰 (action = keep) : Step 3 임베딩에 들어갈 데이터 627,455건 (91.59%)
### 제거리뷰 (action = drop) : 중복으로 판단되어 빠진 데이터. 검증용으로 별도 csv에 보관. 57,587건 (8.41%)
- unique : 가장 많은 케이스. 같은 (작성자, 상품, 24h 세션) 안에 본인 밖에 없는 단독 리뷰. 어떠한 페어 관계도 없으므로 그냥 보존.
- multi_type_unique : 같은 옵션 + 다른 타입 + 텍스트 다름. (한 사람이 같은 사이즈/색을 사고 두가지 후기 타입을 작성했는데 내용이 서로 다른 경우) -> 둘 다 보존
- multi_type_dup : 같은 옵션 + 다른 타입 + 텍스트 사실상 동일. (한 사람이 두가지 후기 양식에 같은 내용을 복붙한 경우) -> 정보량 많은 base는 keep/나머지는 drop
- month_excluded : 한 달 사용 후기. 무조건 보존
- multi_option_unique : 다른 옵션 + 같은 타입 + 텍스트 다름. (한 사람이 같은 상품을 여러 옵션으로 사고 각 옵션에 맞는 다른 후기를 쓴 경우) -> 둘 다 보존
- multi_option_dup : 다른 옵션 + 같은 타입 + 텍스트 사실상 동일 (한 사람이 여러 옵션 사고 같은 후기를 옵션마다 복붙한 경우) -> base만 keep/나머지는 drop
- multi_both_unique : 다른 옵션 + 다른 타입 + 텍스트 다름. (한 사람이 다른 옵션의 다른 양식에 다른 내용 작성) -> 둘 다 보존
- same_option_same_type_dup : 같은 옵션 + 같은 타입 + 24h 이내 (한 사람이 같은 옵션을 사서 같은 타입의 후기를 24h 이내 두 번 이상 작성한 경우. 텍스트 유사도와 무관하게 무조건 1개만 keep) (여기서 keep 조건은 1.정보량, 2.먼저 작성한 리뷰)
- multi_both_dup : 다른 옵션 + 다른 타입 + 텍스트 동일 -> base만 keep

In [18]:
# CSV 저장
OUTPUT_KEEP = "reviews_step2_dedup.csv"
OUTPUT_DROP = "reviews_step2_dropped.csv"

df_keep.to_csv(OUTPUT_KEEP, index=False)
df_drop.to_csv(OUTPUT_DROP, index=False)

print(f"저장 완료:")
print(f"  - {OUTPUT_KEEP} ({len(df_keep):,}건)")
print(f"  - {OUTPUT_DROP} ({len(df_drop):,}건)")

저장 완료:
  - reviews_step2_dedup.csv (627,455건)
  - reviews_step2_dropped.csv (57,587건)


---
## (참고) 검증용 분석

전체 처리 결과를 한눈에 점검하기 위한 통계.
실제 분석에는 사용하지 않으므로 필요 시 셀 단위로 참고.

In [19]:
# 같은 작성자 + 같은 상품인데 옵션이 달랐던 케이스 통계
# (base/동반자 모두 같은 플래그를 공유하므로 kept_ 접두사 없음)
multi_opt_pairs = df_keep[df_keep['dup_flag'].isin([
    'multi_option_unique', 'multi_option_dup'
])]
print(f"옵션 다중 구매 관련 보존된 리뷰: {len(multi_opt_pairs):,}건")
print(f"  - base: {multi_opt_pairs['is_base'].sum():,}건")
print(f"  - 동반자(unique 케이스): {(~multi_opt_pairs['is_base']).sum():,}건")

# 재구매 통계
repurchase_keep = df_keep[df_keep['is_repurchase']]
print(f"\n재구매 케이스 보존 리뷰: {len(repurchase_keep):,}건")
print(f"재구매 (작성자, 상품) 쌍: {repurchase_keep.groupby(['작성자_norm', 'goodsNo']).ngroups:,}쌍")

# 브랜드/카테고리별 처리 결과 (만약 컬럼 있으면)
if '브랜드' in df_keep.columns and '카테고리' in df_keep.columns:
    print(f"\n[브랜드 x 카테고리별 보존 리뷰 수]")
    print(df_keep.groupby(['브랜드', '카테고리']).size().unstack(fill_value=0))

옵션 다중 구매 관련 보존된 리뷰: 8,424건
  - base: 4,676건
  - 동반자(unique 케이스): 3,748건

재구매 케이스 보존 리뷰: 120,749건
재구매 (작성자, 상품) 쌍: 50,181쌍


In [22]:
# dup_flag별 샘플 텍스트 확인
import pandas as pd
pd.set_option('display.max_colwidth', None)

SAMPLE_N = 3        # 각 플래그별 표시할 페어/행 수
TEXT_PREVIEW = 120  # 텍스트 미리보기 길이

def print_pair_samples(flag_name, source_df, n=SAMPLE_N):
    """_dup 또는 _unique 페어 출력. 동반자(is_base=False) 행을 골라 base와 매칭."""
    print(f"\n{'='*95}")
    print(f"[dup_flag = {flag_name}]")
    print('='*95)

    companions = source_df[
        (source_df['dup_flag'] == flag_name) & (~source_df['is_base'].fillna(False))
    ]
    if len(companions) == 0:
        print("  (해당 플래그의 동반자 행이 없음)")
        return

    sampled = companions.sample(n=min(n, len(companions)), random_state=42)

    for k, (_, comp) in enumerate(sampled.iterrows(), 1):
        base_match = source_df[source_df['리뷰번호'] == comp['kept_id']]
        if len(base_match) == 0:
            continue
        base = base_match.iloc[0]

        same_opt = base['옵션키'] == comp['옵션키']
        same_typ = base['리뷰타입'] == comp['리뷰타입']
        print(f"\n  ─ 페어 {k} ─ (옵션 {'동일' if same_opt else '다름'} / "
              f"타입 {'동일' if same_typ else '다름'})")
        print(f"    [base]      리뷰번호={base['리뷰번호']} | dup_flag={base['dup_flag']} | "
              f"is_base={base['is_base']} | action={base['action']}")
        print(f"                옵션={base['옵션키']} | 타입={base['리뷰타입']} | "
              f"한글_글자수={base['한글_글자수']}")
        print(f"                → {str(base[TEXT_COL])[:TEXT_PREVIEW]}")
        print(f"    [동반자]    리뷰번호={comp['리뷰번호']} | dup_flag={comp['dup_flag']} | "
              f"is_base={comp['is_base']} | action={comp['action']}")
        print(f"                옵션={comp['옵션키']} | 타입={comp['리뷰타입']} | "
              f"한글_글자수={comp['한글_글자수']}")
        print(f"                → {str(comp[TEXT_COL])[:TEXT_PREVIEW]}")


def print_solo_samples(flag_name, source_df, n=SAMPLE_N):
    """단독 케이스 (unique, month_excluded) 출력."""
    print(f"\n{'='*95}")
    print(f"[dup_flag = {flag_name}]")
    print('='*95)

    pool = source_df[source_df['dup_flag'] == flag_name]
    if len(pool) == 0:
        print("  (해당 플래그 없음)")
        return

    sampled = pool.sample(n=min(n, len(pool)), random_state=42)
    for k, (_, row) in enumerate(sampled.iterrows(), 1):
        print(f"\n  ─ 샘플 {k} ─")
        print(f"    리뷰번호={row['리뷰번호']} | dup_flag={row['dup_flag']} | "
              f"is_base={row['is_base']} | action={row['action']}")
        print(f"    옵션={row['옵션키']} | 타입={row['리뷰타입']} | "
              f"한글_글자수={row['한글_글자수']}")
        print(f"    → {str(row[TEXT_COL])[:TEXT_PREVIEW]}")


# 동반자(drop된 행 포함)도 매칭에 필요하므로 df_final 사용
source = df_final

# _dup 플래그 (페어 출력, base는 keep / 동반자는 drop)
for flag in ['same_option_same_type_dup', 'multi_type_dup',
             'multi_option_dup', 'multi_both_dup']:
    print_pair_samples(flag, source)

# _unique 플래그 (페어 출력, 둘 다 keep)
for flag in ['multi_type_unique', 'multi_option_unique', 'multi_both_unique']:
    print_pair_samples(flag, source)

# 단독 플래그
for flag in ['unique', 'month_excluded']:
    print_solo_samples(flag, source)


[dup_flag = same_option_same_type_dup]

  ─ 페어 1 ─ (옵션 동일 / 타입 동일)
    [base]      리뷰번호=42421340 | dup_flag=multi_type_dup | is_base=True | action=keep
                옵션=('L', '그레이') | 타입=style | 한글_글자수=26
                → 두번째 구매하는데 만족합니다. 두꺼워서 한여름엔 못입습니다
    [동반자]    리뷰번호=42421392 | dup_flag=same_option_same_type_dup | is_base=False | action=drop
                옵션=('L', '그레이') | 타입=style | 한글_글자수=26
                → 두번째 구매하는데 만족합니다. 두꺼워서 한여름엔 못입습니다

  ─ 페어 2 ─ (옵션 동일 / 타입 동일)
    [base]      리뷰번호=9079594 | dup_flag=same_option_same_type_dup | is_base=True | action=keep
                옵션=('L', '블랙') | 타입=goods | 한글_글자수=20
                → 오버핏으로 크게나왔어요 팔은 좀 길지만 좋네요
    [동반자]    리뷰번호=9079596 | dup_flag=same_option_same_type_dup | is_base=False | action=drop
                옵션=('L', '블랙') | 타입=goods | 한글_글자수=20
                → 오버핏으로 크게나왔어요 팔은 좀 길지만 좋네요

  ─ 페어 3 ─ (옵션 동일 / 타입 동일)
    [base]      리뷰번호=33214788 | dup_flag=same_option_same_type_dup | is_base=True | action=keep
     

In [23]:
# 그룹 크기별 분포 확인
print("[(작성자, 상품, 세션) 그룹 크기 분포]")
print(df_final[df_final['리뷰타입'] != 'month']
      .groupby(['작성자_norm', 'goodsNo', '세션']).size()
      .value_counts().sort_index())

[(작성자, 상품, 세션) 그룹 크기 분포]
1    425294
2     71897
3     28472
4       734
5       102
6       315
7         5
8         2
9         7
Name: count, dtype: int64


In [24]:
# 그룹 크기 ≥ 3인 케이스만 추출해서 그룹 단위로 모든 멤버 표시

LARGE_GROUP_SAMPLE_N = 5  # 출력할 그룹 수

def print_large_group_samples(source_df, min_size=3, n=LARGE_GROUP_SAMPLE_N):
    """그룹 사이즈 ≥ min_size 인 케이스를 그룹 단위로 출력. 모든 멤버를 한 번에 보여줌."""
    
    # month는 별도 처리되므로 제외
    df_check = source_df[source_df['리뷰타입'] != 'month'].copy()
    
    # 그룹 크기 계산
    sizes = df_check.groupby(['작성자_norm', 'goodsNo', '세션']).size()
    large_groups = sizes[sizes >= min_size]
    
    if len(large_groups) == 0:
        print(f"\n그룹 크기 ≥ {min_size}인 케이스 없음")
        return
    
    print(f"\n{'='*95}")
    print(f"[그룹 크기 ≥ {min_size} 케이스: 총 {len(large_groups):,}그룹]")
    print('='*95)
    
    # 그룹 크기 분포
    print("\n그룹 크기별 개수:")
    print(large_groups.value_counts().sort_index().to_string())
    
    # 무작위 n개 그룹 선택
    sample_keys = large_groups.sample(n=min(n, len(large_groups)), 
                                       random_state=42).index.tolist()
    
    for gi, key in enumerate(sample_keys, 1):
        author, goods, sess = key
        members = df_check[
            (df_check['작성자_norm'] == author) &
            (df_check['goodsNo'] == goods) &
            (df_check['세션'] == sess)
        ].sort_values(['is_base', '한글_글자수'], ascending=[False, False])
        
        # 이 그룹 안의 dup_flag 종류 한눈에
        flag_summary = members['dup_flag'].value_counts().to_dict()
        
        print(f"\n{'─'*95}")
        print(f"그룹 {gi}: 작성자={author} | 상품={goods} | 세션={sess} | 멤버 {len(members)}명")
        print(f"  플래그 구성: {flag_summary}")
        print(f"  base 리뷰번호: {members[members['is_base']]['리뷰번호'].iloc[0]}")
        print(f"{'─'*95}")
        
        for mi, (_, m) in enumerate(members.iterrows(), 1):
            role = "★ base" if m['is_base'] else "  동반자"
            print(f"\n  [{role}] 멤버 {mi}")
            print(f"    리뷰번호={m['리뷰번호']} | dup_flag={m['dup_flag']} | "
                  f"is_base={m['is_base']} | action={m['action']}")
            print(f"    옵션={m['옵션키']} | 타입={m['리뷰타입']} | "
                  f"한글_글자수={m['한글_글자수']} | 작성일={m['작성일']}")
            print(f"    → {str(m[TEXT_COL])[:TEXT_PREVIEW]}")


# 그룹 크기별로 따로 보기
print_large_group_samples(source, min_size=3, n=5)   # 3명 이상 그룹 5개
print_large_group_samples(source, min_size=4, n=3)   # 4명 이상 그룹 3개
print_large_group_samples(source, min_size=5, n=2)   # 5명 이상 그룹 2개 (있다면)


[그룹 크기 ≥ 3 케이스: 총 29,637그룹]

그룹 크기별 개수:
3    28472
4      734
5      102
6      315
7        5
8        2
9        7

───────────────────────────────────────────────────────────────────────────────────────────────
그룹 1: 작성자=뮤쉼사 | 상품=1899755 | 세션=1 | 멤버 3명
  플래그 구성: {'multi_type_unique': 3}
  base 리뷰번호: 17989923
───────────────────────────────────────────────────────────────────────────────────────────────

  [★ base] 멤버 1
    리뷰번호=17989923 | dup_flag=multi_type_unique | is_base=True | action=keep
    옵션=('EXTRA LARGE', None) | 타입=photo | 한글_글자수=32 | 작성일=2021-07-03 19:23:29
    → 두깨가 얇지도 두껍지도 않고 적당함 크림색임 약간아이보리 색인데 먼차이냐

  [  동반자] 멤버 2
    리뷰번호=17989900 | dup_flag=multi_type_unique | is_base=False | action=keep
    옵션=('EXTRA LARGE', None) | 타입=style | 한글_글자수=31 | 작성일=2021-07-03 19:22:40
    → 길이가 넣어입고 빼입고 해도 괜찮음 색이 화이트가아니라 크림이라 더 좋음

  [  동반자] 멤버 3
    리뷰번호=17989861 | dup_flag=multi_type_unique | is_base=False | action=keep
    옵션=('EXTRA LARGE', None) | 타입=goods | 한글_글자수=23 | 작성일=202

---
## ⚠️ 알려진 한계: base 행의 dup_flag 의미 모호함

현재 구조에서 그룹 내 여러 동반자가 있고 각 동반자가 서로 다른 dup_flag를 가지는 경우, base 행의 dup_flag는 **그룹 내 첫 번째로 발견된 `_dup` 플래그를 그대로 사용**한다.

### 예시
한 그룹에 멤버 6명이 있고:
- base ↔ 동반자 2: `multi_both_dup` (다른 옵션 + 다른 타입)
- base ↔ 동반자 4: `multi_option_dup` (다른 옵션 + 같은 타입)
- base ↔ 동반자 5: `multi_type_dup` (같은 옵션 + 다른 타입)

이때 base 행의 `dup_flag`는 첫 번째인 `multi_both_dup`으로 표시되지만, 실제로 base는 다른 종류의 관계도 함께 가지고 있다.

### 영향
- **알고리즘 동작에는 문제 없음**: base/동반자 구분은 `is_base` 컬럼이 정확히 처리한다.
- **통계 시 주의**: `df['dup_flag'] == 'multi_both_dup'`로 단순 필터링하면 base와 동반자가 섞이며, base 카운트가 misleading할 수 있다.

### 권장 분석 패턴
페어/플래그 단위 분석 시 **항상 동반자 기준으로 필터링**한다:

```python
# 정확한 multi_both_dup 페어 카운트
df[(df['dup_flag'] == 'multi_both_dup') & (~df['is_base'])]
```

base의 정보가 필요하면 `kept_id`로 매칭하여 가져온다.

### 왜 그대로 두는가
1. 임베딩(Step 3) 입력은 `action == 'keep'`인 모든 행이며, dup_flag 분류 자체에 의존하지 않는다.
2. 분석 단계에서 동반자 기준 필터링 패턴을 지키면 모호함이 발생하지 않는다.

In [29]:
df_keep.head()

,리뷰번호,goodsNo,리뷰타입,작성자,리뷰내용,평점,체험단,구매옵션,키,몸무게,성별,작성일,만족도,사진유무,도움돼요,...,일평균_도움돼요지수,도움여부,리뷰내용_clean,한글_글자수,전체_글자수,is_valid_for_topic,작성자_norm,옵션키,세션,is_repurchase,그룹크기,action,dup_flag,is_base,kept_id
0,34612047,1733275,goods,!!!!!!!!?,요즘 입기 좋은 것 같아요 무난무난하게 잘 입고있습니다,5.0,False,다크그레이 · L,166.0,63.0,여성,2022-11-07 18:13:53,NaN,False,0,...,0.0,0,요즘 입기 좋은 것 같아요 무난무난하게 잘 입고있습니다,23,30,True,!!!!!!!!?,"(L, 다크그레이)",1,False,1,keep,unique,True,34612047
1,51564285,3070563,goods,!!!!!!!!?,잠옷용으로 구매했어요 편하고 그냥 입기에도 조아요,5.0,False,블랙 · 2XL,180.0,90.0,missing,2023-11-23 15:50:39,NaN,False,0,...,0.0,0,잠옷용으로 구매했어요 편하고 그냥 입기에도 조아요,22,27,True,!!!!!!!!?,"(2XL, 블랙)",1,False,1,keep,unique,True,51564285
2,46679669,3251750,goods,!!!!!!!!?,잠옷용으로 휘뚜루마뚜루 입으려고 좀 크게 사긴했는데 진짜 많이 커요 조금은 두꺼운 감이 있어요,5.0,False,블랙 · XL,166.0,50.0,여성,2023-08-01 15:28:26,NaN,False,0,...,0.0,0,잠옷용으로 휘뚜루마뚜루 입으려고 좀 크게 사긴했는데 진짜 많이 커요 조금은 두꺼운 감이 있어요,40,52,True,!!!!!!!!?,"(XL, 블랙)",1,True,1,keep,unique,True,46679669
3,49013088,3251750,goods,!!!!!!!!?,잠옷으로 입으려고 크게 사긴했는데 정말 크고 두꺼워요,5.0,False,블랙 · XL,166.0,50.0,여성,2023-10-01 20:02:46,NaN,False,0,...,0.0,0,잠옷으로 입으려고 크게 사긴했는데 정말 크고 두꺼워요,23,29,True,!!!!!!!!?,"(XL, 블랙)",2,True,1,keep,unique,True,49013088
4,12731504,850153,goods,!!!!!!-,이뻐요. 자주 입겠네요.. 굿 감사랍니다 고맙습니다,5.0,False,블랙 · L,178.0,74.0,남성,2020-11-03 18:57:52,NaN,False,0,...,0.0,0,이뻐요. 자주 입겠네요.. 굿 감사랍니다 고맙습니다,20,28,True,!!!!!!-,"(L, 블랙)",1,False,1,keep,unique,True,12731504
